In [ ]:
# ==========================================
# STEP 1 : Import Libraries
# ==========================================

# Basic Libraries
import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# TensorFlow / Keras
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Embedding,
    LSTM,
    Bidirectional,
    Dense,
    Dropout,
    Layer,
    SpatialDropout1D
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

from tensorflow.keras.optimizers import Adam

# Scikit-Learn
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

# Progress Bar
from tqdm.auto import tqdm

print("TensorFlow Version :", tf.__version__)
print("NumPy Version      :", np.__version__)
print("Pandas Version     :", pd.__version__)

In [ ]:
# ==========================================
# Reproducibility
# ==========================================

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Random Seed Set :", SEED)

In [ ]:
# ==========================================
# Check GPU
# ==========================================

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print(f"✅ GPU Detected : {gpus[0].name}")
else:
    print("❌ GPU Not Detected")

In [ ]:
# ==========================================
# Global Configuration
# ==========================================

MAX_VOCAB_SIZE = 50000      # Maximum vocabulary
MAX_LEN = 384               # Maximum sequence length
EMBEDDING_DIM = 100         # GloVe 100d

BATCH_SIZE = 32
EPOCHS = 20

N_SPLITS = 5

print("Configuration Loaded Successfully")

In [ ]:
# ==========================================
# STEP 2 : Load Dataset
# ==========================================

TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print("=" * 50)
print("Train Shape :", train.shape)
print("Test Shape  :", test.shape)
print("=" * 50)

In [ ]:
# ==========================================
# View Dataset
# ==========================================

print("Train Dataset")
display(train.head())

print("\nTest Dataset")
display(test.head())

In [ ]:
# ==========================================
# Dataset Information
# ==========================================

print("TRAIN INFO")
print("-" * 40)
train.info()

print("\nTEST INFO")
print("-" * 40)
test.info()

In [ ]:
# ==========================================
# Missing Values
# ==========================================

print("Train Missing Values")
display(train.isnull().sum())

print("\nTest Missing Values")
display(test.isnull().sum())

In [ ]:
# ==========================================
# Duplicate Prompts
# ==========================================

duplicates = train["prompt"].duplicated().sum()

print(f"Duplicate Questions : {duplicates}")

In [ ]:
# ==========================================
# Duplicate Prompts
# ==========================================

duplicates = test["prompt"].duplicated().sum()

print(f"Duplicate Questions : {duplicates}")

In [ ]:
# ==========================================
# Target Distribution
# ==========================================

print(train["answer"].value_counts())

plt.figure(figsize=(7,4))

train["answer"].value_counts().sort_index().plot(
    kind="bar"
)

plt.title("Answer Distribution")
plt.xlabel("Answer")
plt.ylabel("Count")
plt.grid(axis="y", alpha=0.3)

plt.show()

In [ ]:
# ==========================================
# Prompt Length
# ==========================================

train["prompt_length"] = train["prompt"].astype(str).apply(len)

print(train["prompt_length"].describe())

plt.figure(figsize=(8,4))

plt.hist(train["prompt_length"], bins=50)

plt.title("Prompt Length Distribution")
plt.xlabel("Characters")
plt.ylabel("Frequency")

plt.show()

In [ ]:
# ==========================================
# Combined Text Length
# ==========================================

train["combined_length"] = (
    train["prompt"].astype(str) +
    train["A"].astype(str) +
    train["B"].astype(str) +
    train["C"].astype(str) +
    train["D"].astype(str) +
    train["E"].astype(str)
).apply(len)

print(train["combined_length"].describe())

plt.figure(figsize=(8,4))

plt.hist(train["combined_length"], bins=50)

plt.title("Combined Text Length")
plt.xlabel("Characters")
plt.ylabel("Frequency")

plt.show()

In [ ]:
# ==========================================
# Label Balance
# ==========================================

label_percent = (
    train["answer"]
    .value_counts(normalize=True)
    .sort_index() * 100
)

display(label_percent)

In [ ]:
# ==========================================
# STEP 3 : Text Cleaning
# ==========================================

import re
import string

def clean_text(text):
    """
    Clean text while preserving sentence meaning.
    """

    if pd.isna(text):
        return ""

    text = str(text)

    # Lowercase
    text = text.lower()

    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove multiple spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [ ]:
# ==========================================
# Clean All Text Columns
# ==========================================

text_columns = ["prompt", "A", "B", "C", "D", "E"]

for col in text_columns:
    train[col] = train[col].apply(clean_text)
    test[col] = test[col].apply(clean_text)

print("Text Cleaning Completed.")

In [ ]:
# ==========================================
# Create Combined Text
# ==========================================

def combine_text(row):

    return (
        "Question: " + row["prompt"] +
        " [SEP] Option A: " + row["A"] +
        " [SEP] Option B: " + row["B"] +
        " [SEP] Option C: " + row["C"] +
        " [SEP] Option D: " + row["D"] +
        " [SEP] Option E: " + row["E"]
    )

train["combined_text"] = train.apply(combine_text, axis=1)
test["combined_text"] = test.apply(combine_text, axis=1)

print("Combined Text Created.")

In [ ]:
print(train["combined_text"].iloc[0])

In [ ]:
train["num_words"] = train["combined_text"].apply(
    lambda x: len(x.split())
)

print(train["num_words"].describe())

In [ ]:
plt.figure(figsize=(8,4))

plt.hist(
    train["num_words"],
    bins=50
)

plt.title("Combined Text Word Count")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")

plt.show()

In [ ]:
print("95th Percentile Length")

print(np.percentile(train["num_words"],95))

In [ ]:
MAX_LEN = 384

In [ ]:
# ==========================================
# STEP 4 : Label Encoding
# ==========================================

from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

train["label"] = label_encoder.fit_transform(train["answer"])

print(label_encoder.classes_)
print(train[["answer", "label"]].head())

In [ ]:
label2id = {label: idx for idx, label in enumerate(label_encoder.classes_)}
id2label = {idx: label for label, idx in label2id.items()}

print(label2id)
print(id2label)

In [ ]:
# ==========================================
# Tokenizer
# ==========================================

tokenizer = Tokenizer(
    num_words=MAX_VOCAB_SIZE,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(train["combined_text"])

print("Vocabulary Size :", len(tokenizer.word_index))

In [ ]:
train_sequences = tokenizer.texts_to_sequences(train["combined_text"])
test_sequences = tokenizer.texts_to_sequences(test["combined_text"])

In [ ]:
print(train["combined_text"].iloc[0])

print("\nSequence:\n")

print(train_sequences[0][:50])

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

X = pad_sequences(
    train_sequences,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_test = pad_sequences(
    test_sequences,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

y = train["label"].values

In [ ]:
print("Training Shape :", X.shape)
print("Test Shape     :", X_test.shape)
print("Labels Shape   :", y.shape)

In [ ]:
print(MAX_LEN)

In [ ]:
sequence_lengths = [len(seq) for seq in train_sequences]

print("Maximum Sequence Length :", max(sequence_lengths))
print("Average Sequence Length :", np.mean(sequence_lengths))
print("95th Percentile         :", np.percentile(sequence_lengths, 95))

In [ ]:
# ==========================================
# STEP 5 : Load GloVe Embeddings
# ==========================================

GLOVE_PATH = "/kaggle/input/datasets/danielwillgeorge/glove6b100dtxt/glove.6B.100d.txt"

print("Using GloVe File:")
print(GLOVE_PATH)

In [ ]:
# ==========================================
# Load GloVe Embeddings
# ==========================================

embeddings_index = {}

with open(GLOVE_PATH, encoding="utf8") as f:

    for line in f:

        values = line.split()

        word = values[0]

        vector = np.asarray(values[1:], dtype="float32")

        embeddings_index[word] = vector

print("Total GloVe Words Loaded :", len(embeddings_index))

In [ ]:
# ==========================================
# Vocabulary Information
# ==========================================

word_index = tokenizer.word_index

print("Vocabulary Size :", len(word_index))

In [ ]:
# ==========================================
# Create Embedding Matrix
# ==========================================

vocab_size = min(MAX_VOCAB_SIZE, len(word_index) + 1)

embedding_matrix = np.zeros((vocab_size, EMBEDDING_DIM))

for word, idx in word_index.items():

    if idx >= vocab_size:
        continue

    vector = embeddings_index.get(word)

    if vector is not None:
        embedding_matrix[idx] = vector

In [ ]:
# ==========================================
# Check Coverage
# ==========================================

found = 0

for word, idx in word_index.items():

    if idx >= vocab_size:
        continue

    if word in embeddings_index:
        found += 1

coverage = found / vocab_size

print(f"Vocabulary Covered : {found}")
print(f"Coverage           : {coverage:.2%}")

In [ ]:
print("Embedding Matrix Shape")

print(embedding_matrix.shape)

In [ ]:
word = "python"

if word in word_index:

    idx = word_index[word]

    print("Index :", idx)

    print("Embedding Vector:")

    print(embedding_matrix[idx][:15])

In [ ]:
# ==========================================
# STEP 6 : Custom Attention Layer
# ==========================================

from tensorflow.keras.layers import Layer
import tensorflow.keras.backend as K

class AttentionLayer(Layer):

    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):

        self.W = self.add_weight(
            name="attention_weight",
            shape=(input_shape[-1], 1),
            initializer="glorot_uniform",
            trainable=True,
        )

        self.b = self.add_weight(
            name="attention_bias",
            shape=(input_shape[1], 1),
            initializer="zeros",
            trainable=True,
        )

        super(AttentionLayer, self).build(input_shape)

    def call(self, inputs):

        # Alignment scores
        e = K.tanh(K.dot(inputs, self.W) + self.b)

        # Attention weights
        a = K.softmax(e, axis=1)

        # Weighted hidden states
        output = inputs * a

        # Context Vector
        context = K.sum(output, axis=1)

        return context

In [ ]:
dummy = tf.random.normal((2, 256, 128))

attention = AttentionLayer()

output = attention(dummy)

print(output.shape)

In [ ]:
# ==========================================
# STEP 7 : Build BiLSTM + Attention Model
# ==========================================

from tensorflow.keras.layers import (
    Input,
    Embedding,
    Bidirectional,
    LSTM,
    Dense,
    Dropout,
    SpatialDropout1D
)
from tensorflow.keras.models import Model

def build_model():

    inputs = Input(shape=(MAX_LEN,), name="Input")

    # ---------------------------------------------------
    # Embedding Layer (Pretrained GloVe)
    # ---------------------------------------------------
    x = Embedding(
        input_dim=vocab_size,
        output_dim=EMBEDDING_DIM,
        weights=[embedding_matrix],
        input_length=MAX_LEN,
        trainable=False,          # Freeze GloVe initially
        name="Embedding"
    )(inputs)

    x = SpatialDropout1D(0.3)(x)

    # ---------------------------------------------------
    # First BiLSTM
    # ---------------------------------------------------
    x = Bidirectional(
        LSTM(
            128,
            return_sequences=True,
            dropout=0.3,
            recurrent_dropout=0.3
        )
    )(x)

    # ---------------------------------------------------
    # Second BiLSTM
    # ---------------------------------------------------
    x = Bidirectional(
        LSTM(
            64,
            return_sequences=True,
            dropout=0.3,
            recurrent_dropout=0.3
        )
    )(x)

    # ---------------------------------------------------
    # Attention Layer
    # ---------------------------------------------------
    x = AttentionLayer()(x)

    # ---------------------------------------------------
    # Dense Layers
    # ---------------------------------------------------
    x = Dense(256, activation="relu")(x)

    x = Dropout(0.4)(x)

    x = Dense(128, activation="relu")(x)

    x = Dropout(0.3)(x)

    outputs = Dense(
        5,
        activation="softmax",
        name="Output"
    )(x)

    model = Model(inputs, outputs)

    return model

In [ ]:
model = build_model()

In [ ]:
model.summary()

In [ ]:
# ==========================================
# STEP 8 : Compile Model
# ==========================================

from tensorflow.keras.optimizers import Adam

In [ ]:
model.compile(
    optimizer=Adam(
        learning_rate=1e-3
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)

In [ ]:
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

In [ ]:
early_stop = EarlyStopping(

    monitor="val_loss",

    patience=3,

    restore_best_weights=True,

    verbose=1
)

In [ ]:
reduce_lr = ReduceLROnPlateau(

    monitor="val_loss",

    factor=0.5,

    patience=2,

    min_lr=1e-6,

    verbose=1
)

In [ ]:
checkpoint = ModelCheckpoint(

    "best_model.keras",

    monitor="val_loss",

    save_best_only=True,

    verbose=1
)

In [ ]:
callbacks = [

    early_stop,

    reduce_lr,

    checkpoint
]

In [ ]:
model.summary()

In [ ]:
# ==========================================
# STEP 9 : 5 Fold Cross Validation
# ==========================================

from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

In [ ]:
# Validation predictions
oof_pred = np.zeros((len(train), 5))

# Test predictions
test_pred = np.zeros((len(test), 5))

In [ ]:
fold_accuracy = []
fold_losses = []

In [ ]:
for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):

    print("="*60)
    print(f"Fold {fold}")
    print("="*60)

    X_train = X[train_idx]
    X_valid = X[valid_idx]

    y_train = y[train_idx]
    y_valid = y[valid_idx]

In [ ]:
    model = build_model()

    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

In [ ]:
    history = model.fit(

        X_train,
        y_train,

        validation_data=(X_valid, y_valid),

        epochs=20,

        batch_size=BATCH_SIZE,

        callbacks=callbacks,

        verbose=1
    )

In [ ]:
    valid_prob = model.predict(
        X_valid,
        verbose=0
    )

    oof_pred[valid_idx] = valid_prob

In [ ]:
    fold_test = model.predict(
        X_test,
        verbose=0
    )

    test_pred += fold_test / skf.n_splits

In [ ]:
    pred_label = np.argmax(valid_prob, axis=1)

    acc = accuracy_score(
        y_valid,
        pred_label
    )

    fold_accuracy.append(acc)

    print(f"Fold {fold} Accuracy : {acc:.4f}")

In [ ]:
print("="*60)

print("Cross Validation Finished")

print("="*60)

In [ ]:
overall_pred = np.argmax(oof_pred, axis=1)

overall_acc = accuracy_score(
    y,
    overall_pred
)

print(f"\nOverall Accuracy : {overall_acc:.4f}")

In [ ]:
# ==========================================
# STEP 10 : Top-3 Predictions
# ==========================================

top3_indices = np.argsort(test_pred, axis=1)[:, -3:]

In [ ]:
top3_indices = top3_indices[:, ::-1]

In [ ]:
predictions = []

for row in top3_indices:

    labels = [id2label[i] for i in row]

    predictions.append(" ".join(labels))

In [ ]:
submission = pd.DataFrame({

    "id": test["id"],

    "Prediction": predictions

})

In [ ]:
submission.head()

In [ ]:
submission.to_csv(

    "submission.csv",

    index=False
)

print("Submission Saved Successfully!")

In [ ]:
submission.sample(10)

In [ ]:
def mapk(actual, predicted, k=3):

    score = 0.0

    for a, p in zip(actual, predicted):

        if a in p[:k]:

            score += 1.0 / (p.index(a) + 1)

    return score / len(actual)

In [ ]:
oof_top3 = np.argsort(oof_pred, axis=1)[:, -3:][:, ::-1]

oof_labels = []

for row in oof_top3:
    oof_labels.append([id2label[i] for i in row])

actual = train["answer"].tolist()

score = mapk(actual, oof_labels, k=3)

print(f"Validation MAP@3 : {score:.5f}")